### Simple Chain
---
A 3 steps chain:
1. Take input from user.
2. Pass the input to a LLM.
3. Show the LLM's output to the user.

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

HUGGINGFACE_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACE_API_TOKEN

In [ ]:
# make the prompt
prompt = PromptTemplate(
    template = "Generate 5 interesting fact about {topic}",
    input_variables = ['topic']
)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    task="text-generation",
    # huggingfacehub_api_token = HUGGINGFACE_API_TOKEN
)
model = ChatHuggingFace(llm = llm)

In [ ]:
# make a string parser
from langchain_core.output_parsers import StrOutputParser
sting_parser = StrOutputParser()

In [ ]:
# form the chain
simple_chain = prompt | model | sting_parser

In [ ]:
user_topic = input("Enter a topic name: ")
output = simple_chain.invoke({
    "topic": user_topic
})
print(output)

In [ ]:
simple_chain.get_graph().print_ascii()

**Building a 2 step sequential Chain**

- Take a topic from user.
- Make a prompt to generate a detailed report on it.
- Pass the topic to a LLM and generate a detailed report on it.
- Make a 2nd prompt to get 5 most import point from the detailed report.
- Send it to the same LLM and get the key 5 points.

In [ ]:
# define the prompt
prompt1 = PromptTemplate(
    template = "Generate a detailed report on this {topic}.",
    input_variables = ["topic"]
)
prompt2 = PromptTemplate(
    template = "Give me the keywords from the given report here.\n {report}",
    input_variables = ['report']
)

In [ ]:
# define 
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()

In [ ]:
model

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
# define the chain
chain1 = (
    prompt1| 
    model| 
    str_parser| 
    RunnableLambda(lambda x: {"report" : x})|
    prompt2| 
    model|
    str_parser
)

In [ ]:
user_topic = "Unemployment issue in Bangladesh."
result = chain1.invoke({
    "topic": user_topic
})

In [ ]:
type(result)

In [ ]:
print(result)

In [ ]:
# visualize the chain1
chain1.get_graph().print_ascii()

### Building a Parallel Chain.
* Take a long detailed text(explaination of something).
* Parallel Part:
   - Send to model-1 to generate notes from the doc.
   - Send to model-2 to generate quiz(mcq) from the doc.
* Take the note and quiz then send these to a 3rd model to combine these.

In [ ]:
model_for_note = ChatHuggingFace(
    llm = HuggingFaceEndpoint(
        repo_id = "meta-llama/Llama-3.2-1B-Instruct",
        task = "text-generation",
        max_new_tokens=1024
    )
)
model_for_quiz = ChatHuggingFace(
    llm = HuggingFaceEndpoint(
        repo_id="microsoft/Phi-3-mini-4k-instruct",
        task = "text-generation",
        max_new_tokens=1024
    )
)

In [ ]:
prompt_for_note = PromptTemplate(
    template = "Generate short and simple note from the following text: \n {text}",
    input_variables = ['text']
)
prompt_for_quiz = PromptTemplate(
    template = "Generate few multiple choice questions from the following text: \n{text}",
    input_variables = ['text']
)
prompt_for_merge_note_quiz = PromptTemplate(
    template = """Merge the note and multiple choice question into a single document.\n
    Note: {note} \n Multiple-Choice Questions: {questions}
    """,
    input_variables = ["note" , "questions"]
)

In [ ]:
# define the parser
str_parser

In [ ]:
from langchain_core.runnables import RunnableParallel

In [ ]:
# Define 2 sequential chain
note_chain = prompt_for_note | model_for_note | str_parser
quiz_chain = prompt_for_quiz | model | str_parser

In [ ]:
# now parallel these 2 sequential chain  in parallel
parallel_chain = RunnableParallel(
    note = note_chain,
    questions = quiz_chain
)

In [ ]:
# marge with final model
final_chain = parallel_chain | prompt_for_merge_note_quiz | model | str_parser

In [ ]:
with open("doc.txt" , "r") as f:
    doc = f.read()

doc

In [ ]:
result = final_chain.invoke({
    "text": doc
})

In [ ]:
print(result)

In [ ]:
final_chain.get_graph().print_ascii()

### Conditional Chain

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "text-generation",
    temperature = 0.1,
    max_new_tokens = 512
)
model = ChatHuggingFace(llm = llm)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Sentiment(BaseModel):
    sentiment_type: Literal['positive' , 'negative'] = Field(
        description = "Give the sentiment of the feedback"
    )

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
pydantic_parser = PydanticOutputParser(
    pydantic_object = Sentiment
)

In [ ]:
prompt1 = PromptTemplate(
    template = """Classify the sentiment of the following feedback text
    into only either positive or negative in the following format.\n {feedback}\n
    Format: {format_instruction}""",
    input_variables = ['feedback'],
    partial_variables = {
        "format_instruction" : pydantic_parser.get_format_instructions()
    }
)

classifier = prompt1 | model | pydantic_parser

In [ ]:
review_text = """
Sarah here. Absolutely love this coffee maker! Brews perfectly every time.
Super easy to clean and looks great on the counter. 
The only downside is it's quite loud and the carafe lid is flimsy.
"""
print(
    classifier.invoke({
        "feedback": review_text
    })
)

In [ ]:
sentiment_result = classifier.invoke({
    "feedback": review_text
})

In [ ]:
type(sentiment_result)

In [ ]:
sentiment_result.sentiment_type

In [ ]:
# Step 1: carry feedback alongside sentiment result
from langchain_core.runnables import RunnablePassthrough
classifier_chain = RunnablePassthrough.assign(
    sentiment = lambda x: classifier.invoke({"feedback": x["feedback"]})
)

In [ ]:
pos_prompt = PromptTemplate(
    template = "Write an appropriate email for this positive feedback:\n{feedback}",
    input_variables = ["feedback"]
)
neg_prompt = PromptTemplate(
    template = "Write an appropriate email for this negative feedback:\n{feedback}",
    input_variables = ["feedback"]
)

In [ ]:
# now based on the sentiment make 2 branch: +ve and -ve branch
from langchain_core.runnables import RunnableBranch

# Step 2: branch based on sentiment, pass feedback to prompts
branch_chain = RunnableBranch(
    (
        lambda x: x["sentiment"].sentiment_type == "positive",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | pos_prompt | model | str_parser
    ),
    (
        lambda x: x["sentiment"].sentiment_type == "negative",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | neg_prompt | model | str_parser
    ),
    RunnableLambda(lambda x: "Could not find any sentiment.")
)

In [ ]:
# Step 3: full chain
full_chain = classifier_chain | branch_chain

In [ ]:
result = full_chain.invoke({"feedback": review_text})
print(result)